In [ ]:
# ==========================================
# 0. SETUP & DEPENDENCIES
# ==========================================
!pip install -q x-transformers flash-attn datasets pandas tabulate huggingface_hub

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import gc
import pandas as pd
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from x_transformers import TransformerWrapper, Encoder

# Global Config
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATASET_ID = "prism-lab/wikitext-103-prism-test-seed42"
BATCH_SIZE = 8
VOCAB_SIZE = 32768
SEQ_LEN = 4096

print(f"🔥  Initializing Full Benchmark Suite on {DEVICE}")

# ==========================================
# 1. ARCHITECTURE DEFINITIONS (Exact Matches)
# ==========================================

# --- UTILS (Shared) ---
class ComplexDropout(nn.Module):
    def __init__(self, p=0.0): super().__init__(); self.p = p
    def forward(self, z): return z
class RobustPhaseNorm(nn.Module):
    def __init__(self, d, eps=1e-5): super().__init__(); self.scale = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x): return (x / torch.sqrt((x.abs()**2).mean(-1, keepdim=True) + self.eps)) * self.scale
class ModReLU(nn.Module):
    def __init__(self, f): super().__init__(); self.b = nn.Parameter(torch.zeros(f))
    def forward(self, z): return F.relu(z.abs() + self.b) * (z / (z.abs() + 1e-6))
class ComplexToRealBridge(nn.Module):
    def __init__(self, d): super().__init__(); self.proj = nn.Linear(d*2, d); self.norm = nn.LayerNorm(d)
    def forward(self, x): return self.norm(self.proj(torch.cat([x.real, x.imag], -1)))
class DynamicRoSE(nn.Module):
    def __init__(self, n, d):
        super().__init__(); self.raw_embedding = nn.Embedding(n, d); self.adapter = nn.Linear(d, d*2); self.rotation_predictor = nn.Linear(d, d*2)
        self.register_buffer('freqs', torch.exp(torch.arange(0, d) * -(math.log(10000.0)/d)))
    def forward(self, x):
        real = self.raw_embedding(x); params = self.adapter(real); D = real.shape[-1]
        z = torch.complex(params[...,:D], params[...,D:]); r = self.rotation_predictor(real); rx, ry = r.chunk(2, -1)
        drot = torch.complex(rx/torch.sqrt(rx**2+ry**2+1e-6), ry/torch.sqrt(rx**2+ry**2+1e-6))
        pos = torch.arange(real.shape[1], device=x.device).float()
        srot = torch.polar(torch.ones_like(torch.outer(pos, self.freqs)), torch.outer(pos, self.freqs))
        return (z * srot.unsqueeze(0) * drot), real
class HyenaNeuralFilter(nn.Module):
    def __init__(self, d, max_len=1024, h=64):
        super().__init__(); self.d = d; self.register_buffer("freqs", torch.exp(torch.arange(0, h, 2) * -(math.log(10000.0)/h)))
        self.mlp = nn.Sequential(nn.Linear(h, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, d*2))
    def forward(self, L, dev):
        t = torch.linspace(0, 1, steps=L, device=dev).unsqueeze(-1)
        emb = torch.cat([torch.sin(t*self.freqs), torch.cos(t*self.freqs)], -1)
        out = self.mlp(emb).view(L, self.d, 2); return torch.complex(out[...,0], out[...,1])
class GatedHarmonicConvolution(nn.Module):
    def __init__(self, d, max_len):
        super().__init__(); self.d=d; self.filter_len=max_len; self.neural_filter = HyenaNeuralFilter(d, max_len)
        self.gate_proj = nn.Linear(d*2, d*2); self.mix_real = nn.Linear(d,d); self.mix_imag = nn.Linear(d,d)
        self.out_real = nn.Linear(d,d); self.out_imag = nn.Linear(d,d); self.activation = ModReLU(d); self.norm = RobustPhaseNorm(d)
        self.dropout = ComplexDropout(0.0)
    def forward(self, x, mask=None):
        res = x; x = self.norm(x); B,L,D = x.shape; eff_L = min(L, self.filter_len)
        h = self.neural_filter(eff_L, x.device).unsqueeze(0)
        xt = torch.fft.ifft(torch.fft.fft(x, n=eff_L, dim=1, norm='ortho') * h, n=eff_L, dim=1, norm='ortho')
        if L > eff_L: xt = F.pad(xt, (0,0,0,L-eff_L));
        else: xt = xt[:, :L, :]
        g = torch.sigmoid(self.gate_proj(torch.cat([x.real, x.imag], -1))); gr, gi = g.chunk(2, -1)
        xg = torch.complex(xt.real*gr, xt.imag*gi); mr, mi = self.mix_real, self.mix_imag
        xm = torch.complex(mr(xg.real)-mi(xg.imag), mr(xg.imag)+mi(xg.real)); xa = self.activation(xm); or_, oi = self.out_real, self.out_imag
        out = torch.complex(or_(xa.real)-oi(xa.imag), or_(xa.imag)+oi(xa.real))
        return self.dropout(out) + res
class PRISMEncoder(nn.Module):
    def __init__(self, l, d, max_l): super().__init__(); self.layers = nn.ModuleList([GatedHarmonicConvolution(d, max_l) for _ in range(l)]); self.final_norm = RobustPhaseNorm(d)
    def forward(self, x):
        for layer in self.layers: x = layer(x)
        return self.final_norm(x)

# --- A. BASELINE (Transformer) ---
class LocalBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = TransformerWrapper(
            num_tokens=VOCAB_SIZE, max_seq_len=SEQ_LEN, use_abs_pos_emb=False, tie_embedding=True,
            attn_layers=Encoder(dim=512, depth=5, heads=8, rotary_pos_emb=True, attn_flash=True, use_scalenorm=False)
        )
    def forward(self, x): return self.model(x)

# --- B. FNET (Hybrid) ---
class FNetBlock(nn.Module):
    def __init__(self, d, df):
        super().__init__(); self.norm_mix = nn.LayerNorm(d); self.norm_ff = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, df), nn.GELU(), nn.Dropout(0), nn.Linear(df, d), nn.Dropout(0))
    def forward(self, x):
        r = x; x = self.norm_mix(x); x = torch.fft.fftn(x.float(), dim=(-2,-1), norm='ortho').real.to(r.dtype); x = x+r
        r = x; x = self.norm_ff(x); x = self.ff(x); return x+r
class FNetEncoder(nn.Module):
    def __init__(self, depth, d, df): super().__init__(); self.layers = nn.ModuleList([FNetBlock(d, df) for _ in range(depth)]); self.norm_out = nn.LayerNorm(d)
    def forward(self, x):
        for l in self.layers: x = l(x)
        return self.norm_out(x)
class HybridFNetMLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_emb = nn.Embedding(VOCAB_SIZE, 512); self.pos_emb = nn.Parameter(torch.zeros(1, SEQ_LEN, 512))
        self.fnet_encoder = FNetEncoder(6, 512, 2048)
        self.transformer_cap = Encoder(dim=512, depth=1, heads=8, rotary_pos_emb=True, attn_flash=True)
        self.final_norm = nn.LayerNorm(512); self.to_logits = nn.Linear(512, VOCAB_SIZE)
        self.to_logits.weight = self.token_emb.weight # Tie
    def forward(self, x):
        h = self.token_emb(x) + self.pos_emb[:, :x.shape[1], :]
        return self.to_logits(self.final_norm(self.transformer_cap(self.fnet_encoder(h))))

# --- C. PRISM (Phase Coder) ---
class LocalPRISM(nn.Module):
    def __init__(self):
        super().__init__()
        self.rose = DynamicRoSE(VOCAB_SIZE, 512); self.prism_encoder = PRISMEncoder(5, 512, SEQ_LEN)
        self.bridge = ComplexToRealBridge(512); self.periscope_proj = nn.Sequential(nn.Linear(1024, 512), nn.LayerNorm(512), nn.GELU())
        self.refiner = Encoder(dim=512, depth=1, heads=8, rotary_pos_emb=True, attn_flash=True)
        self.lm_head = nn.Linear(512, VOCAB_SIZE); self.lm_head.weight = self.rose.raw_embedding.weight # Tie
    def forward(self, x):
        w, p = self.rose(x); w = self.bridge(self.prism_encoder(w))
        return self.lm_head(self.refiner(self.periscope_proj(torch.cat([w, p], -1))))

# --- D. PILLARS (Split-Stream) ---
class LocalPillars(nn.Module):
    def __init__(self):
        super().__init__()
        self.rose = DynamicRoSE(VOCAB_SIZE, 512); self.particle_down = nn.Linear(512, 256); self.wave_down = nn.Linear(1024, 512)
        self.fnet_pos = nn.Embedding(SEQ_LEN, 256); self.stream_rate = FNetEncoder(9, 256, 1024)
        self.stream_phase = PRISMEncoder(9, 256, SEQ_LEN); self.phase_bridge = ComplexToRealBridge(256)
        self.fusion_proj = nn.Linear(512, 512); self.fusion_norm = nn.LayerNorm(512)
        self.refiner = Encoder(dim=512, depth=1, heads=8, rotary_pos_emb=True, attn_flash=True)
        self.head_bias = nn.Parameter(torch.zeros(VOCAB_SIZE))
    def forward(self, x):
        w, p = self.rose(x); p_sm = self.particle_down(p); w_raw = self.wave_down(torch.cat([w.real, w.imag], -1))
        w_sm = torch.complex(w_raw[...,:256], w_raw[...,256:])
        p_path = self.stream_rate(p_sm + self.fnet_pos(torch.arange(x.shape[1], device=x.device)))
        w_path = self.phase_bridge(self.stream_phase(w_sm))
        ctx = self.fusion_norm(self.fusion_proj(torch.cat([p_path, w_path], -1)))
        return F.linear(self.refiner(ctx), self.rose.raw_embedding.weight, self.head_bias)

# --- NEW: Sensory Stream for DAT ---
class SensoryStream(nn.Module):
    def __init__(self, depth, d, dropout=0.1):
        super().__init__()
        self.encoder = Encoder(
            dim=d, depth=depth, heads=4, attn_flash=True,
            rotary_pos_emb=True, attn_dropout=dropout, ff_dropout=dropout,
            use_rmsnorm=True, ff_glu=True
        )
    def forward(self, x): return self.encoder(x)

# --- E. PILLARS-DAT (New Hybrid) ---
class LocalPillarsDAT(nn.Module):
    def __init__(self):
        super().__init__()
        # Config matching your training
        d_model, d_branch, depth = 512, 256, 6

        self.rose = DynamicRoSE(VOCAB_SIZE, d_model)
        self.particle_down = nn.Linear(d_model, d_branch)
        self.wave_down = nn.Linear(d_model * 2, d_branch * 2)

        # Stream A: Sensory (Transformer)
        self.stream_sensory = SensoryStream(depth, d_branch)

        # Stream B: Relational (PRISM)
        # Re-using existing PRISMEncoder(layers, dim, seq_len)
        self.stream_relational = PRISMEncoder(depth, d_branch, SEQ_LEN)
        self.relational_bridge = ComplexToRealBridge(d_branch)

        # Fusion
        self.fusion_proj = nn.Linear(d_branch * 2, d_model)
        self.fusion_norm = nn.LayerNorm(d_model)

        # Refiner
        self.refiner = Encoder(dim=d_model, depth=1, heads=8, rotary_pos_emb=True, attn_flash=True)

        # Output (Standard Linear to match HF Checkpoint)
        self.lm_head = nn.Linear(d_model, VOCAB_SIZE)
        self.lm_head.weight = self.rose.raw_embedding.weight # Tie

    def forward(self, x):
        w, p = self.rose(x)
        p_sm = self.particle_down(p)

        # Complex Downsample
        w_raw = self.wave_down(torch.cat([w.real, w.imag], -1))
        w_sm = torch.complex(w_raw[...,:256], w_raw[...,256:])

        # Parallel Streams
        sensory_out = self.stream_sensory(p_sm)
        rel_out = self.relational_bridge(self.stream_relational(w_sm))

        # Fusion
        ctx = self.fusion_norm(self.fusion_proj(torch.cat([sensory_out, rel_out], -1)))
        return self.lm_head(self.refiner(ctx))

# ==========================================
# 2. INTELLIGENT LOADING & EVALUATION
# ==========================================
def smart_load(model, repo_id, name):
    print(f"⬇️  Downloading {name} from HF: {repo_id}...")
    try: path = hf_hub_download(repo_id, "best.pt")
    except: path = hf_hub_download(repo_id, "pytorch_model.bin")

    state_dict = torch.load(path, map_location="cpu")
    if 'model' in state_dict: state_dict = state_dict['model']

    clean = {k.replace("module.", ""): v for k, v in state_dict.items()}

    # 🔧 SPECIFIC FIXES (Programmatic remapping)
    if name == "Baseline":
        new_d = {}
        for k, v in clean.items():
            nk = k if k.startswith("model.") else "model." + k
            if "token_emb.weight" in nk and "emb" not in nk: nk = nk.replace("token_emb.weight", "token_emb.emb.weight")
            new_d[nk] = v
        clean = new_d
    elif name == "FNet":
        new_d = {}
        for k, v in clean.items():
            nk = k.replace("model.", "")
            new_d[nk] = v
        clean = new_d

    # LOAD
    missing, _ = model.load_state_dict(clean, strict=False)
    print(f"✅  {name} Loaded. Missing Keys: {len(missing)}")
    return model

def evaluate_full(model, loader):
    model.eval()
    total_nll = 0
    total_mask_count = 0 # Track exact number of predictions
    correct_1 = 0
    correct_5 = 0

    # Switch to SUM reduction to handle varying mask counts correctly
    criterion = nn.CrossEntropyLoss(reduction='sum')

    with torch.no_grad():
        for b in tqdm(loader, leave=False):
            ids = b['input_ids'].to(DEVICE)
            lbl = b['labels'].to(DEVICE)

            logits = model(ids)
            if isinstance(logits, dict): logits = logits['logits']

            # Mask logic: only calculate loss on -100 (masked) tokens
            mask = (lbl != -100)
            if mask.sum() > 0:
                # 1. PPL: Sum the loss, don't mean it yet
                loss = criterion(logits.view(-1, VOCAB_SIZE), lbl.view(-1))
                total_nll += loss.item()
                total_mask_count += mask.sum().item()

                # 2. Accuracy
                m_log = logits[mask]
                m_lbl = lbl[mask]
                correct_1 += (m_log.argmax(-1) == m_lbl).sum().item()
                _, top5 = m_log.topk(5, -1)
                correct_5 += (top5 == m_lbl.unsqueeze(1)).any(1).sum().item()

    if total_mask_count == 0: return 0, 0, 0

    # Final Calculation
    ppl = math.exp(total_nll / total_mask_count)
    acc1 = (correct_1 / total_mask_count) * 100
    acc5 = (correct_5 / total_mask_count) * 100
    return ppl, acc1, acc5


def audit_efficiency(model, name):
    total = sum(p.numel() for p in model.parameters())
    is_tied = total < 45000000 # Hard check for 30-40M range

    active_mix = 0
    if name == "Baseline":
        for n, p in model.named_parameters():
            if "attn" in n and "norm" not in n: active_mix += p.numel()
    elif name == "FNet":
        for n, p in model.named_parameters():
            if "transformer_cap" in n and "attn" in n and "norm" not in n: active_mix += p.numel()
    elif name == "PRISM":
        for n, p in model.named_parameters():
            if ("neural_filter" in n or "gate_proj" in n) or ("refiner" in n and "attn" in n and "norm" not in n):
                active_mix += p.numel()
    elif name == "PILLARS": # Old HSSM
        for n, p in model.named_parameters():
            if (("neural_filter" in n or "gate_proj" in n) and "stream_phase" in n) or ("fusion_proj" in n) or ("refiner" in n and "attn" in n and "norm" not in n):
                active_mix += p.numel()
    # --- NEW BLOCK ---
    elif name == "PILLARS-DAT":
        for n, p in model.named_parameters():
            # 1. Sensory Stream Attention (Rate)
            if "stream_sensory" in n and "attn" in n and "norm" not in n:
                active_mix += p.numel()
            # 2. Relational Stream Filters (Phase)
            if "stream_relational" in n and ("neural_filter" in n or "gate_proj" in n):
                active_mix += p.numel()
            # 3. Refiner Attention (Readout)
            if "refiner" in n and "attn" in n and "norm" not in n:
                active_mix += p.numel()

    return {
        "Model": name,
        "Weights Tied?": "✅ YES" if is_tied else "❌ NO",
        "Total Params (M)": total / 1e6,
        "Active Mixing (M)": active_mix / 1e6,
        "Mixing Ratio": f"{(active_mix/total)*100:.1f}%"
    }
# ==========================================
# 4. EXECUTION
# ==========================================
MODELS = [
    ("Baseline", "prism-lab/baseline-wikitext-prism", LocalBaseline),
    ("FNet", "prism-lab/hybrid-fnet-prism-custom", HybridFNetMLM),
    ("PRISM", "prism-lab/prism-v2-wikitext", LocalPRISM),
    ("HSSM", "prism-lab/pillars-compact-wikitext", LocalPillars),
    ("WPT", "prism-lab/pillars-dat-wikitext-32k", LocalPillarsDAT)
]
print("📦  Loading Data...")
d = load_dataset(DATASET_ID)

# Priority: Test > Validation > Train (fallback)
if 'test' in d:
    ds = d['test']
elif 'validation' in d:
    ds = d['validation']
else:
    ds = d['train']

print(f"📊  Evaluating on split: {ds.split if hasattr(ds, 'split') else 'Unknown'}")
ds.set_format("torch", columns=["input_ids", "labels"])
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)

perf_results = []
param_results = []

print("\n🔥  STARTING FULL BENCHMARK")

for name, repo, cls in MODELS:
    print(f"\n🧪  Processing {name}...")
    try:
        model = cls().to(DEVICE)
        model = smart_load(model, repo, name)

        # 1. Performance
        ppl, top1, top5 = evaluate_full(model, loader)
        perf_results.append({"Model": name, "PPL": ppl, "Top-1": top1, "Top-5": top5})

        # 2. Efficiency
        param_results.append(audit_efficiency(model, name))

        del model; torch.cuda.empty_cache(); gc.collect()
    except Exception as e:
        print(f"❌  {name} Failed: {e}")

print("\n\n" + "="*80)
print("🏆 TABLE 1: PERFORMANCE BENCHMARK")
print("="*80)
print(pd.DataFrame(perf_results).sort_values("PPL").to_markdown(index=False, floatfmt=".4f"))

print("\n\n" + "="*80)
print("🏆 TABLE 2: COMPUTATIONAL EFFICIENCY")
print("="*80)
print(pd.DataFrame(param_results).to_markdown(index=False, floatfmt=".2f"))